# 파이썬 과제 풀이

1. 필요한 패키지를 불러들인다
2. 역주소 정보가 담긴 CSV 파일을 OPEN 한다
3. 국토교통부의 OPEN API 데이터를 가져온다 (테스트)
   (명세서 확인 -요청 정보 확인 -웹 데이터 요청 - 결과 확인)
   해당 과정을 재사용성을 위해 함수로 생성한다

4. 위도와 경도 데이터를 추출한 후 데이터를 정제한다 (동기식과 비동기식)
5. 기존 데이터 셋에 위도와 경도 데이터를 함께 붙이고 엑셀 형태로 저장한다

### STEP1. 패키지 참조

In [1]:
import requests
from pandas import DataFrame
from concurrent import futures

### STEP2. CSV 파일 읽어오기

In [2]:
#서울 열린 데이터 광장 - 서울 교통공사 역주소 및 전화번호 최근 데이터셋을 다운받아 r 읽기 모드로 데이터를 가져온다
with open("서울교통공사_역주소 및 전화번호_20250318.csv","r",encoding='euc-kr') as f:
  #데이터 세트 객체를 한 행씩 읽는 readlines() 메서드를 활용하고, 해당 결과를 csv_list 에 담는다
  csv_list=f.readlines()

print(csv_list[:5])

['연번,역번호,호선,역명,역전화번호,도로명주소,지번주소\n', '1,150,1,서울,02-6110-1331,서울특별시 중구 세종대로 지하2(남대문로 5가),서울특별시 중구 남대문로5가 73-6 서울역(1호선)\n', '2,151,1,시청,02-6110-1321,서울특별시 중구 세종대로 지하101(정동),서울특별시 중구 정동 5-5 시청역(1호선)\n', '3,152,1,종각,02-6110-1311,서울특별시 종로구 종로 지하55(종로1가),서울특별시 종로구 종로1가 54 종각역(1호선)\n', '4,153,1,종로3가,02-6110-1301,서울특별시 종로구 종로 지하129(종로3가),서울특별시 종로구 종로3가 10-5 종로3가역(1호선)\n']


### STEP 3. Open API 데이터 요청 스펙 확인
OPEN API 를 연결할 때의 작업 항목들 살펴보면,
OPEN API 요청 정보를 확인하고, SESSION 객체로 URL 에 접속하여 데이터를 가져오는 것.


In [3]:
#요청 url
url = "https://api.vworld.kr/req/address?"
key = "1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E"

#요청 파라미터 (명세서 확인)
params = {
	"service": "address",
	"request": "getcoord",
	"crs": "epsg:4326",
	"address": "판교로 242",
	"format": "json",
	"type": "road",
	"key": key
}

In [ ]:
#웹에 데이터 요청하기
with requests.Session() as session:

  #세션 객체에 웹 브라우저 정보 (UserAgent) 주입 (웹서버가 파이썬 프로그램을 정상적인 웹 브라우저로 여기도록)
  session.headers.update({"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.103 Safari/537.36"
                          })
  

  r=session.get(url,params=params)    #get 메서드로 접근하고, 파라미터 값 쿼리 스트링 형태로 전달
  print(r.url)

  if r.status_code !=200:
    msg ="[%d Error] %s 에러가 발생함" % (r.status_code,r.reason)
    raise Exception(msg)

print(r) # HTTP 통신 상태 확인


https://api.vworld.kr/req/address?service=address&request=getcoord&crs=epsg%3A4326&address=%ED%8C%90%EA%B5%90%EB%A1%9C+242&format=json&type=road&key=1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E
<Response [200]>


In [10]:
#요청 테스트 확인
mydict = r.json()
mydict

{'response': {'service': {'name': 'address',
   'version': '2.0',
   'operation': 'getcoord',
   'time': '20(ms)'},
  'status': 'OK',
  'input': {'type': 'road', 'address': '판교로 242'},
  'refined': {'text': '경기도 성남시 분당구 판교로 242 (삼평동)',
   'structure': {'level0': '대한민국',
    'level1': '경기도',
    'level2': '성남시 분당구',
    'level3': '삼평동',
    'level4L': '판교로',
    'level4LC': '',
    'level4A': '삼평동',
    'level4AC': '4113565500',
    'level5': '242',
    'detail': ''}},
  'result': {'crs': 'EPSG:4326',
   'point': {'x': '127.101313354', 'y': '37.402352535'}}}}

In [16]:
#위도와 경도 추출

point=mydict["response"]["result"]["point"]
print(point["x"],point["y"])

127.101313354 37.402352535


### STEP4. 위경도를 조회하는 함수 정의
위에서 테스트로 확인한 과정을 함수로 정의

In [22]:
addr = "서울시 관악구 봉천동 1534-28"

def get_point(addr,type ="road"):
  #요청 url
  url = "https://api.vworld.kr/req/address?"
  key = "1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E"

  #요청 파라미터 (명세서 확인)
  params = {
    "service": "address",
    "request": "getcoord",
    "crs": "epsg:4326",
    "address": addr,
    "format": "json",
    "type": type,
    "key": key
  }


#웹에 데이터 요청하기
  with requests.Session() as session:

    #세션 객체에 웹 브라우저 정보 (UserAgent) 주입 (웹서버가 파이썬 프로그램을 정상적인 웹 브라우저로 여기도록)
    session.headers.update({"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.103 Safari/537.36"
                            })
    

    r=session.get(url,params=params)    #get 메서드로 접근하고, 파라미터 값 쿼리 스트링 형태로 전달
    print(r.url)

    if r.status_code !=200:
      msg ="[%d Error] %s 에러가 발생함" % (r.status_code,r.reason)
      raise Exception(msg)

    mydict = r.json()
    point=mydict["response"]["result"]["point"]
      
    return (point["x"],point["y"])


### STEP.5-1 위경도 변환하기 | 동기 방식
- OPEN API 를 활용하기 위해서는 주소지 정보를 넣어 경도와 위도 값을 반환해야하고,
해당 과정을 get_point() 라는 함수로 정의한 상황. 이 함수를 쓰려면 주소 정보를 get_point() 함수에 파라미터로 전달해야 하므로, 우선 csv_list 에서 주소 정보를 추출한다. 
- 추출한 주소 정보를 get_point() 함수에 넣으면 경도와 위도 정보가 나오는데, 만약 기본값으로 세팅한 type=road 에서 변환이 되지 않았다면 , 지번 주소로 재시도 하는 예외 처리를 걸어준다
- 만약 지번 주소에서도 예외로 처리되면 그때는 위도와 경도에 None 값을 넣어준다
- 기존에 존재하던 csv_list 데이터의 각 항목들에 위도와 경도를 추가해주고, 이 결과를 비워진 리스트 resultset에 넣어주면 리스트 내에 각 데이터들이 또 다른 리스트 형태로 존재하는 2차원 구조의 데이터가 완성된다

In [ ]:
size = len(csv_list)
resultset=[]
for i,v in enumerate(csv_list[1:11]): #제목을 건너뛰기 위해 1부터
  print("%d/%d 진행중..."%(i+1,size))
  items=v.strip().split(",")

  try:
    lat,lon = get_point(items[5])
  
  except Exception as e :
    try:
      lat,lon =get_point(items[6],type="parcel")

    except Exception as e2:
      lat =None
      lon=None

  items.append(lat)
  items.append(lon)
  resultset.append(items)


resultset

1/290 진행중...
https://api.vworld.kr/req/address?service=address&request=getcoord&crs=epsg%3A4326&address=%EC%84%9C%EC%9A%B8%ED%8A%B9%EB%B3%84%EC%8B%9C+%EC%A4%91%EA%B5%AC+%EC%84%B8%EC%A2%85%EB%8C%80%EB%A1%9C+%EC%A7%80%ED%95%982%28%EB%82%A8%EB%8C%80%EB%AC%B8%EB%A1%9C+5%EA%B0%80%29&format=json&type=road&key=1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E
https://api.vworld.kr/req/address?service=address&request=getcoord&crs=epsg%3A4326&address=%EC%84%9C%EC%9A%B8%ED%8A%B9%EB%B3%84%EC%8B%9C+%EC%A4%91%EA%B5%AC+%EB%82%A8%EB%8C%80%EB%AC%B8%EB%A1%9C5%EA%B0%80+73-6+%EC%84%9C%EC%9A%B8%EC%97%AD%281%ED%98%B8%EC%84%A0%29&format=json&type=parcel&key=1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E
2/290 진행중...
https://api.vworld.kr/req/address?service=address&request=getcoord&crs=epsg%3A4326&address=%EC%84%9C%EC%9A%B8%ED%8A%B9%EB%B3%84%EC%8B%9C+%EC%A4%91%EA%B5%AC+%EC%84%B8%EC%A2%85%EB%8C%80%EB%A1%9C+%EC%A7%80%ED%95%98101%28%EC%A0%95%EB%8F%99%29&format=json&type=road&key=1C1C7AC9-4A35-3854-BFE7-D91FAEDD655E
3/290 진행중...
http

[['1',
  '150',
  '1',
  '서울',
  '02-6110-1331',
  '서울특별시 중구 세종대로 지하2(남대문로 5가)',
  '서울특별시 중구 남대문로5가 73-6 서울역(1호선)',
  '126.97254323694754',
  '37.55702928798156'],
 ['2',
  '151',
  '1',
  '시청',
  '02-6110-1321',
  '서울특별시 중구 세종대로 지하101(정동)',
  '서울특별시 중구 정동 5-5 시청역(1호선)',
  '126.978346780',
  '37.566700969'],
 ['3',
  '152',
  '1',
  '종각',
  '02-6110-1311',
  '서울특별시 종로구 종로 지하55(종로1가)',
  '서울특별시 종로구 종로1가 54 종각역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['4',
  '153',
  '1',
  '종로3가',
  '02-6110-1301',
  '서울특별시 종로구 종로 지하129(종로3가)',
  '서울특별시 종로구 종로3가 10-5 종로3가역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['5',
  '154',
  '1',
  '종로5가',
  '02-6110-1291',
  '서울특별시 종로구 종로 지하216(종로5가)',
  '서울특별시 종로구 종로5가 82-1 종로5가역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['6',
  '155',
  '1',
  '동대문',
  '02-6110-1281',
  '서울특별시 종로구 종로 지하302(창신동)',
  '서울특별시 종로구 창신동 492-1 동대문역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['7',
  '156',
  '1',
  '신설동',
  '02-6110-1261',
  '서울특별시 동대문구 왕산로 지하1(신설동)',
 

### STEP.5-2 위경도 변환하기 | 비동기 방식
- 동기식에 사용한 코드를 그대로 가져와 futures.ThreadPoolExecutor 로 감싼다
- 프로세스 과정과 결과를 넣을 빈 리스트를 만들어 둔다

In [32]:
size = len(csv_list)
resultset=[]
processes=[] # 비동기 작업 프로세스를 저장할 리스트


with futures.ThreadPoolExecutor(max_workers=30) as executor:

  for i,v in enumerate(csv_list[1:101]):     #제목을 건너뛰기 위해 1부터
    print("%d/%d 진행중..."%(i+1,size))
    items=v.strip().split(",")

    pro=executor.submit(get_point,items[5])
    processes.append(pro)   #위에서 준비한 리스트에 저장



  for i,p in enumerate(processes) :   #비동기 프로세스가 실행되는 동안 발생하는 예외에 대비하기 위한 처리 

    try:
      lat,lon = p.result()
    
    except Exception as e :
      try:
        lat,lon =get_point(items[6],type="parcel")

      except Exception as e2:
        lat =None
        lon=None

    items=csv_list[i].strip().split(",")
    items.append(lat)
    items.append(lon)
    resultset.append(items)


resultset

1/290 진행중...
2/290 진행중...
3/290 진행중...
4/290 진행중...
5/290 진행중...
6/290 진행중...
7/290 진행중...
8/290 진행중...
9/290 진행중...
10/290 진행중...
11/290 진행중...
12/290 진행중...
13/290 진행중...
14/290 진행중...
15/290 진행중...
16/290 진행중...
17/290 진행중...
18/290 진행중...
19/290 진행중...
20/290 진행중...
21/290 진행중...
22/290 진행중...
23/290 진행중...
24/290 진행중...
25/290 진행중...
26/290 진행중...
27/290 진행중...
28/290 진행중...
29/290 진행중...
30/290 진행중...
31/290 진행중...
32/290 진행중...
33/290 진행중...
34/290 진행중...
35/290 진행중...
36/290 진행중...
37/290 진행중...
38/290 진행중...
39/290 진행중...
40/290 진행중...
41/290 진행중...
42/290 진행중...
43/290 진행중...
44/290 진행중...
45/290 진행중...
46/290 진행중...
47/290 진행중...
48/290 진행중...
49/290 진행중...
50/290 진행중...
51/290 진행중...
52/290 진행중...
53/290 진행중...
54/290 진행중...
55/290 진행중...
56/290 진행중...
57/290 진행중...
58/290 진행중...
59/290 진행중...
60/290 진행중...
61/290 진행중...
62/290 진행중...
63/290 진행중...
64/290 진행중...
65/290 진행중...
66/290 진행중...
67/290 진행중...
68/290 진행중...
69/290 진행중...
70/290 진행중...
71/290 진행중...
72/290 진행중...
7

[['연번',
  '역번호',
  '호선',
  '역명',
  '역전화번호',
  '도로명주소',
  '지번주소',
  '127.02722259341354',
  '37.639822439425885'],
 ['1',
  '150',
  '1',
  '서울',
  '02-6110-1331',
  '서울특별시 중구 세종대로 지하2(남대문로 5가)',
  '서울특별시 중구 남대문로5가 73-6 서울역(1호선)',
  '126.978346780',
  '37.566700969'],
 ['2',
  '151',
  '1',
  '시청',
  '02-6110-1321',
  '서울특별시 중구 세종대로 지하101(정동)',
  '서울특별시 중구 정동 5-5 시청역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['3',
  '152',
  '1',
  '종각',
  '02-6110-1311',
  '서울특별시 종로구 종로 지하55(종로1가)',
  '서울특별시 종로구 종로1가 54 종각역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['4',
  '153',
  '1',
  '종로3가',
  '02-6110-1301',
  '서울특별시 종로구 종로 지하129(종로3가)',
  '서울특별시 종로구 종로3가 10-5 종로3가역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['5',
  '154',
  '1',
  '종로5가',
  '02-6110-1291',
  '서울특별시 종로구 종로 지하216(종로5가)',
  '서울특별시 종로구 종로5가 82-1 종로5가역(1호선)',
  '127.022379505',
  '37.573411520'],
 ['6',
  '155',
  '1',
  '동대문',
  '02-6110-1281',
  '서울특별시 종로구 종로 지하302(창신동)',
  '서울특별시 종로구 창신동 492-1 동대문역(1호선)',
  '127.0250937

### STEP.5 변환 결과 저장하기


In [33]:
new_resultset = []

for r in resultset:
    item_dict = {
        "역번": r[0],
        "역번호": r[1],
        "호선": r[2],
        "역명": r[3],
        "역전화번호": r[4],
        "도로명주소": r[5],
        "지번주소": r[6],
        "위도": r[7],
        "경도": r[8]
    }

    new_resultset.append(item_dict)

df = DataFrame(new_resultset)
df.to_excel("지하철역.xlsx")
df


,역번,역번호,호선,역명,역전화번호,도로명주소,지번주소,위도,경도
0,연번,역번호,호선,역명,역전화번호,도로명주소,지번주소,127.02722259341354,37.639822439425885
1,1,150,1,서울,02-6110-1331,서울특별시 중구 세종대로 지하2(남대문로 5가),서울특별시 중구 남대문로5가 73-6 서울역(1호선),126.978346780,37.566700969
2,2,151,1,시청,02-6110-1321,서울특별시 중구 세종대로 지하101(정동),서울특별시 중구 정동 5-5 시청역(1호선),127.022379505,37.573411520
3,3,152,1,종각,02-6110-1311,서울특별시 종로구 종로 지하55(종로1가),서울특별시 종로구 종로1가 54 종각역(1호선),127.022379505,37.573411520
4,4,153,1,종로3가,02-6110-1301,서울특별시 종로구 종로 지하129(종로3가),서울특별시 종로구 종로3가 10-5 종로3가역(1호선),127.022379505,37.573411520
...,...,...,...,...,...,...,...,...,...
95,95,409,4,불암산,02-6110-4091,서울특별시 노원구 상계로 305(상계동),서울특별시 노원구 상계동 111 불암산역(4호선),127.073641730,37.660976439
96,96,410,4,상계,02-6110-4101,서울특별시 노원구 상계로 182(상계동),서울특별시 노원구 상계동 156-203 상계역(4호선),127.063028584,37.656267243
97,97,411,4,노원,02-6110-4111,서울특별시 노원구 상계로 69-1(상계동),서울특별시 노원구 상계동 602-5 노원역(4호선),127.047683782,37.653208466
98,98,412,4,창동,02-6110-4121,서울특별시 도봉구 마들로11길 77(창동),서울특별시 도봉구 창동 135-1 창동역(4호선),127.039661274,37.656555835
